# Step 6 - Building Feature Batches (batch_XXX.npz)

This step assembles encoded videos into `.npz` files that match the author's format and
composition:

- `data`: `(10, 100, 20, 4096)`, float16
- `labels`: `(10, 2)`, float16, one-hot `[no-accident, accident]`

Each batch contains a **random 3-5 accident videos** and the rest non-accident (10 total), in
random order. The dataset is imbalanced (610 accident versus 1130 non-accident videos), so the
accident pool is oversampled (cycled across batches) to cover all batches.

Output is written to `Dataset/features_generated/`, leaving the author's reference files
unchanged.

## Models, helpers, and per-video encoder

Consolidated from Steps 2-5.

In [ ]:
import cv2, time, glob, os
import numpy as np
import torch
import torch.nn as nn

N_FRAMES=100; N_DETECTION=20; N_OBJECTS=19; FEAT_DIM=4096
KEEP={1,2,3,4,6,7,8}; SCORE_THRESH=0.5
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
IMAGENET_STD =torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

from torchvision.models.detection import fasterrcnn_resnet50_fpn
try:
    from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
    detector = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
except Exception:
    detector = fasterrcnn_resnet50_fpn(pretrained=True)
detector.eval()

from torchvision.models import vgg16
try:
    from torchvision.models import VGG16_Weights
    vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
except Exception:
    vgg = vgg16(pretrained=True)
vgg.eval()
vgg_features=vgg.features; vgg_avgpool=vgg.avgpool
vgg_fc7=nn.Sequential(*list(vgg.classifier.children())[:5])

def detect_boxes(frame):
    t=torch.from_numpy(frame).permute(2,0,1).float()/255.0
    with torch.no_grad():
        out=detector([t])[0]
    kept=[(float(s),b) for b,l,s in zip(out['boxes'].numpy(),out['labels'].numpy(),out['scores'].numpy())
          if s>=SCORE_THRESH and int(l) in KEEP]
    kept.sort(key=lambda z:z[0], reverse=True)
    return [b for _,b in kept[:N_OBJECTS]]

def crop_box(frame, box):
    h,w,_=frame.shape; x1,y1,x2,y2=box
    x1=max(0,int(round(x1))); y1=max(0,int(round(y1)))
    x2=min(w,int(round(x2))); y2=min(h,int(round(y2)))
    return None if (x2-x1<2 or y2-y1<2) else frame[y1:y2,x1:x2]

def to_features_batch(crops):
    if not crops: return np.zeros((0,FEAT_DIM),np.float32)
    batch=np.stack([cv2.resize(c,(224,224)) for c in crops]).astype(np.float32)/255.0
    x=torch.from_numpy(batch).permute(0,3,1,2)
    x=(x-IMAGENET_MEAN)/IMAGENET_STD
    with torch.no_grad():
        x=vgg_features(x); x=vgg_avgpool(x); x=torch.flatten(x,1); x=vgg_fc7(x)
    return x.numpy().astype(np.float32)

def encode_video(path, max_frames=100):
    cap=cv2.VideoCapture(path); frames=[]
    while True:
        ok,f=cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(f,cv2.COLOR_BGR2RGB))
    cap.release()
    out=np.zeros((N_FRAMES,N_DETECTION,FEAT_DIM),np.float32)
    for t in range(min(max_frames,len(frames))):
        frame=frames[t]; crops=[frame]
        for box in detect_boxes(frame):
            c=crop_box(frame,box)
            if c is not None: crops.append(c)
            if len(crops)==N_DETECTION: break
        feats=to_features_batch(crops)
        out[t,:feats.shape[0],:]=feats
    return out
print('ready.')

## Settings (set once)
Set `VIDEOS_ROOT` / `OUT_ROOT` once here. The split, number of batches and frames are passed to `build_features()`, so you don't edit this cell per run.

In [ ]:
# ----- settings: set once -----
VIDEOS_ROOT = '../Dataset/videos'
OUT_ROOT    = '../Dataset/features_generated'
ACC_MIN, ACC_MAX = 3, 5      # accidents per batch (the rest are non-accident)
SEED = 1234                   # fixed seed -> reproducible AND resumable
# -------------------------------
import random

for sp in ['train', 'test']:
    na = len(glob.glob(f'{VIDEOS_ROOT}/{sp}/Accident/*.mp4'))
    nn = len(glob.glob(f'{VIDEOS_ROOT}/{sp}/NonAccident/*.mp4'))
    print(f'{sp}: {na} accident, {nn} non-accident videos')

## Define `build_features(split, num_batches, max_frames)`
One call builds a whole split. Resumable (already-saved batches are skipped) and deterministic per batch index.

In [ ]:
def build_features(split, num_batches, max_frames, batch_from=1):
    """Build `num_batches` .npz files for `split`, each video encoded with `max_frames` frames.
    Each batch = random ACC_MIN..ACC_MAX accidents + rest non-accident, random order.
    Resumable (already-saved batches are skipped). Example: build_features('test', 10, 100)."""
    acc_pool = sorted(glob.glob(f'{VIDEOS_ROOT}/{split}/Accident/*.mp4'))
    non_pool = sorted(glob.glob(f'{VIDEOS_ROOT}/{split}/NonAccident/*.mp4'))
    assert acc_pool and non_pool, f'no videos for split={split} - check VIDEOS_ROOT'

    def _n_acc(b):
        return random.Random(SEED + b).randint(ACC_MIN, ACC_MAX)

    def plan_batch(b):            # 0-based; deterministic -> resumable
        rng   = random.Random(SEED + b)
        n_acc = rng.randint(ACC_MIN, ACC_MAX)
        n_non = 10 - n_acc
        acc_off = sum(_n_acc(j) for j in range(b))
        non_off = sum(10 - _n_acc(j) for j in range(b))
        acc = [acc_pool[(acc_off + k) % len(acc_pool)] for k in range(n_acc)]
        non = [non_pool[(non_off + k) % len(non_pool)] for k in range(n_non)]
        items = [(p, (0.0, 1.0)) for p in acc] + [(p, (1.0, 0.0)) for p in non]
        rng.shuffle(items)
        return items

    out_dir = f'{OUT_ROOT}/{split}'
    os.makedirs(out_dir, exist_ok=True)
    for b in range(batch_from, num_batches + 1):
        out_path = f'{out_dir}/batch_{b:03d}.npz'
        if os.path.exists(out_path):
            print(f'skip batch_{b:03d} (already done)'); continue
        items  = plan_batch(b - 1)
        data   = np.zeros((10, N_FRAMES, N_DETECTION, FEAT_DIM), np.float32)
        labels = np.zeros((10, 2), np.float32)
        t0 = time.time(); print(f'=== {split} batch_{b:03d} ===')
        for i, (p, lab) in enumerate(items):
            ti = time.time(); data[i] = encode_video(p, max_frames); labels[i] = lab
            print(f'  [{i+1:2d}/10] {"A" if lab[1]==1 else "n"} {os.path.basename(p):12s} {time.time()-ti:5.1f}s')
        np.savez(out_path, data=data.astype(np.float16), labels=labels.astype(np.float16))
        print(f'  saved {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)  total {time.time()-t0:.1f}s\n')
    print(f'DONE: {split} ({num_batches} batches)')
    return out_dir

## Run
Call it with the split, number of batches and frames you want, e.g. `build_features('test', 10, 100)`.

In [ ]:
# build one split in one call:  build_features(split, num_batches, max_frames)
build_features('test', 1, 20)        # quick local test: 1 batch, 20 frames

# full runs (slow on CPU):
#   build_features('test', 10, 100)
#   build_features('train', 20, 100)

## Verify a saved batch

In [ ]:
def verify(split, b=1):
    f = np.load(f'{OUT_ROOT}/{split}/batch_{b:03d}.npz')
    labels = f['labels']; n_acc = int(sum(r[1] == 1 for r in labels))
    print('data  :', f['data'].shape, f['data'].dtype)
    print('labels:', labels.shape)
    print('composition:', n_acc, 'accident +', 10 - n_acc, 'non-accident')
    print('order :', ['A' if r[1] == 1 else 'n' for r in labels])
    print('data >= 0 ?', bool((f['data'] >= 0).all()))

verify('test', 1)

## Summary

- Each batch has a random 3-5 accidents and the rest non-accident, in random order
  (configurable via `ACC_MIN`/`ACC_MAX`).
- Deterministic per batch index (seeded), so reruns and resumed Kaggle commits are reproducible.
- Build a split with one call, e.g. `build_features('train', 20, 100)`.